<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 4 · Unsupervised Learning and Real Projects · Lesson 23 of 24

# Saving, Sharing and Monitoring Models

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Save a trained pipeline with joblib, load it to make predictions, document it with a model card, and check new data for drift.

## A model is only useful when it is used

A notebook model helps nobody. To be useful it must be **saved**, **loaded** by an app (a website, a USSD service, a mobile app) and **watched** over time. This step is called **deployment**, and the wider practice is **MLOps**.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 2000
income = rng.lognormal(12.2, 0.45, n).round(-3)
loan = rng.uniform(20_000, 300_000, n).round(-3)
months = rng.integers(1, 61, n)
late = rng.poisson(0.6, n)
salary = rng.random(n) < 0.55
z = -3.2 + 2.5 * (loan / income) - 0.03 * months + 0.8 * late - 0.9 * salary
default = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
loans = pd.DataFrame({"income_rwf": income, "loan_rwf": loan, "months_customer": months,
                      "late_payments": late, "salaried": salary.astype(int),
                      "defaulted": default})
print(loans.shape, "default rate:", round(loans["defaulted"].mean(), 3))
loans.head()

## Train the final pipeline on all development data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X = loans.drop(columns="defaulted")
y = loans["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0))
pipe.fit(X_train, y_train)
test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print("Test AUC:", round(test_auc, 3))

Save the **whole pipeline**, not just the model — the scaler is part of the model. Saving only the model is a classic bug: the app then feeds it unscaled numbers.

## Save and load with joblib

In [ ]:
import joblib

joblib.dump(pipe, "loan_model_v1.joblib")

loaded = joblib.load("loan_model_v1.joblib")
same = (loaded.predict_proba(X_test) == pipe.predict_proba(X_test)).all()
print("Loaded model gives identical predictions:", same)

In Colab, download the file from the Files panel or save it to Google Drive.

Rules for model files:

- Load files only from sources you **trust** — a joblib/pickle file can run code when opened.
- Record the **scikit-learn version**: a model saved with one version may not load in a very different one.
- Use clear names with versions: `loan_model_v1.joblib`, `v2`…

## A prediction function for the app

In [ ]:
import pandas as pd

FEATURES = ["income_rwf", "loan_rwf", "months_customer", "late_payments", "salaried"]
THRESHOLD = 0.25

def decide(application: dict) -> dict:
    row = pd.DataFrame([application])[FEATURES]
    risk = float(loaded.predict_proba(row)[0, 1])
    return {"risk": round(risk, 3),
            "decision": "refer to officer" if risk >= THRESHOLD else "approve"}

print(decide({"income_rwf": 180_000, "loan_rwf": 90_000, "months_customer": 24,
              "late_payments": 0, "salaried": 1}))
print(decide({"income_rwf": 120_000, "loan_rwf": 250_000, "months_customer": 2,
              "late_payments": 3, "salaried": 0}))

Notice the decision for risky cases is **"refer to officer"**, not "refuse": a person reviews them. A web framework such as **FastAPI** (which powers the WAMUGII website) can wrap a function like this as an API.

## Write a model card

A **model card** is a short document that travels with the model:

In [ ]:
import json
import sklearn
from datetime import date

card = {
    "name": "Micro-loan default risk",
    "version": "v1",
    "date": str(date.today()),
    "sklearn_version": sklearn.__version__,
    "intended_use": "Rank mobile micro-loan applications by default risk; risky cases go to a loan officer.",
    "not_for": "Automatic refusal without human review; loans above 300,000 RWF.",
    "training_data": "2,000 simulated past borrowers (practice data).",
    "features": FEATURES,
    "threshold": THRESHOLD,
    "test_auc": round(test_auc, 3),
    "fairness_check": "Refusal rates audited by region (see Lesson 22).",
    "retrain": "Every 3 months, or when drift is detected.",
}
with open("loan_model_v1_card.json", "w") as f:
    json.dump(card, f, indent=2)
print(json.dumps(card, indent=2)[:400])

## Monitor for drift

The world changes: inflation raises incomes and loan sizes, a new competitor changes who applies. When new data looks different from the training data — **data drift** — the model's accuracy can quietly fall.

In [ ]:
new_month = X_test.copy()
new_month["loan_rwf"] = (new_month["loan_rwf"] * 1.4).round(-3)    # loans grew 40%

comparison = pd.DataFrame({"training mean": X_train.mean(), "this month": new_month.mean()})
comparison["change %"] = ((comparison["this month"] / comparison["training mean"] - 1) * 100).round(1)
print(comparison.round(0))

before = (loaded.predict_proba(X_test)[:, 1] >= THRESHOLD).mean()
after = (loaded.predict_proba(new_month)[:, 1] >= THRESHOLD).mean()
print("Share referred: test data", round(before, 3), "-> this month", round(after, 3))

Loan sizes jumped about 40%, and the share of applicants referred to an officer rises from about a third to about half. That is an alarm: investigate, collect new labelled outcomes, and **retrain**.

What to monitor every month:

1. Input features — averages, ranges, missing values.
2. Predictions — the share approved or referred.
3. Real outcomes once known — recall, precision, AUC.
4. Fairness by group.

## Key terms

- **Deployment** — putting a model into real use.
- **joblib** — a library to save and load Python objects such as pipelines.
- **Model card** — a short document describing a model's use, data, performance and limits.
- **Data drift** — new data that differs from the training data.
- **MLOps** — the practice of deploying, monitoring and maintaining models.

**Takeaway:** Save the whole pipeline with joblib, wrap it in a simple prediction function with human review, document it in a model card and watch for drift so you know when to retrain.

## Try it yourself

1. Save the homes pipeline from Lesson 21, load it, and predict the rent of three new homes.
2. Add `"limitations"` and `"contact"` fields to the model card.
3. Simulate a month where incomes rise 20% and loan sizes stay the same. Does the referral rate go up or down? Why?
4. Write a one-paragraph monitoring plan for the SACCO: what you check, how often, and what triggers retraining.

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*